In [3]:
import pandas as pd
import numpy as np

# -----------------------------
# Load data
# -----------------------------
results = pd.read_csv("Resources/results.csv")
drivers = pd.read_csv("Resources/drivers.csv")
constructors = pd.read_csv("Resources/constructors.csv")
races = pd.read_csv("Resources/races.csv")
qualifying = pd.read_csv("Resources/qualifying.csv")
weather = pd.read_csv("Resources/weather.csv")

# -----------------------------
# 1. Understand datasets
# -----------------------------
datasets = {
    "results": results,
    "drivers": drivers,
    "constructors": constructors,
    "races": races,
    "qualifying": qualifying,
    "weather": weather
}

for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print(df.shape)
    print(df.columns.tolist())
    print(df.head(3))

# -----------------------------
# 2. Unique counts from results
# -----------------------------
print("\nUnique counts from results")
print("Constructors:", results["constructorId"].nunique())
print("Drivers:", results["driverId"].nunique())
print("Races:", results["raceId"].nunique())

# -----------------------------
# 3. Unique counts and mismatch checks
# -----------------------------
def id_check(base_ids, compare_ids, label):
    missing_in_compare = set(base_ids) - set(compare_ids)
    extra_in_compare = set(compare_ids) - set(base_ids)
    print(f"\n{label}")
    print("Missing in compare:", len(missing_in_compare))
    print("Extra in compare:", len(extra_in_compare))

id_check(results["driverId"].unique(), drivers["driverId"].unique(), "Driver ID check")
id_check(results["constructorId"].unique(), constructors["constructorId"].unique(), "Constructor ID check")
id_check(results["raceId"].unique(), races["raceId"].unique(), "Race ID check")

print("\nDuplicate ID checks")
print("drivers duplicate driverId:", drivers["driverId"].duplicated().sum())
print("constructors duplicate constructorId:", constructors["constructorId"].duplicated().sum())
print("races duplicate raceId:", races["raceId"].duplicated().sum())
print("results duplicate resultId:", results["resultId"].duplicated().sum())

# -----------------------------
# 4. Nationality frequency
# -----------------------------
driver_nat_freq = drivers["nationality"].value_counts(dropna=False).reset_index()
driver_nat_freq.columns = ["nationality", "driver_count"]

constructor_nat_freq = constructors["nationality"].value_counts(dropna=False).reset_index()
constructor_nat_freq.columns = ["nationality", "constructor_count"]

print("\nTop driver nationalities")
print(driver_nat_freq.head(10))

print("\nTop constructor nationalities")
print(constructor_nat_freq.head(10))

# -----------------------------
# 5. Prepare qualifying data
# -----------------------------
qual_cols = [c for c in qualifying.columns if c.lower() in ["raceid", "driverid", "position"]]
qual_small = qualifying[qual_cols].copy()
qual_small = qual_small.rename(columns={"position": "quali_position"})

# -----------------------------
# 6. Categorize weather
# -----------------------------
weather["weather_raw"] = weather["weather_raw"].fillna("").astype(str).str.lower()

def classify_weather(x):
    if any(k in x for k in ["rain", "wet", "drizzle", "shower", "storm", "mist"]):
        if any(k in x for k in ["drying", "later dry", "variable", "intermittent"]):
            return "variable"
        return "wet"
    if any(k in x for k in ["dry", "sun", "clear", "hot", "warm", "mild", "cloudy", "overcast"]):
        return "dry"
    return "unknown"

weather["weather_cat"] = weather["weather_raw"].apply(classify_weather)

# -----------------------------
# 7. Merge master data
# -----------------------------
master = results.merge(
    races[["raceId", "year", "round", "circuitId", "name", "date"]],
    on="raceId", how="left"
)

master = master.merge(
    drivers[["driverId", "driverRef", "forename", "surname", "nationality"]],
    on="driverId", how="left"
).rename(columns={"nationality": "driver_nationality"})

master = master.merge(
    constructors[["constructorId", "constructorRef", "name", "nationality"]],
    on="constructorId", how="left", suffixes=("", "_constructor")
).rename(columns={
    "name_constructor": "constructor_name",
    "nationality": "constructor_nationality"
})

master = master.merge(
    qual_small,
    on=["raceId", "driverId"], how="left"
)

master = master.merge(
    weather[["raceId", "weather_raw", "weather_cat"]],
    on="raceId", how="left"
)

# -----------------------------
# 8. Derive target variable
# -----------------------------
master["top3_flag"] = np.where(master["positionOrder"] <= 3, 1, 0)

# -----------------------------
# 9. Constructor analysis
# -----------------------------
constructor_top3 = (
    master.groupby("constructorId")
    .agg(
        constructor_name=("constructor_name", "first"),
        races=("raceId", "nunique"),
        entries=("resultId", "count"),
        top3_finishes=("top3_flag", "sum"),
        top3_rate=("top3_flag", "mean")
    )
    .reset_index()
    .sort_values(["top3_finishes", "top3_rate"], ascending=[False, False])
)

# -----------------------------
# 10. Driver analysis
# -----------------------------
master["driver_name"] = master["forename"].fillna("") + " " + master["surname"].fillna("")

driver_top3 = (
    master.groupby("driverId")
    .agg(
        driver_name=("driver_name", "first"),
        races=("raceId", "nunique"),
        entries=("resultId", "count"),
        top3_finishes=("top3_flag", "sum"),
        top3_rate=("top3_flag", "mean")
    )
    .reset_index()
    .sort_values(["top3_finishes", "top3_rate"], ascending=[False, False])
)

# -----------------------------
# 11. Save outputs
# -----------------------------
master.to_csv("output/phase1_master_data.csv", index=False)
constructor_top3.to_csv("output/phase1_constructor_top3_summary.csv", index=False)
driver_top3.to_csv("output/phase1_driver_top3_summary.csv", index=False)
driver_nat_freq.to_csv("output/phase1_driver_nationality_freq.csv", index=False)
constructor_nat_freq.to_csv("output/phase1_constructor_nationality_freq.csv", index=False)

print("\nPhase 1 complete. Files saved.")


RESULTS
(26080, 18)
['resultId', 'raceId', 'driverId', 'constructorId', 'number', 'grid', 'position', 'positionText', 'positionOrder', 'points', 'laps', 'time', 'milliseconds', 'fastestLap', 'rank', 'fastestLapTime', 'fastestLapSpeed', 'statusId']
   resultId  raceId  driverId  constructorId number  grid position  \
0         1      18         1              1     22     1        1   
1         2      18         2              2      3     5        2   
2         3      18         3              3      7     7        3   

  positionText  positionOrder  points  laps         time milliseconds  \
0            1              1    10.0    58  1:34:50.616      5690616   
1            2              2     8.0    58       +5.478      5696094   
2            3              3     6.0    58       +8.163      5698779   

  fastestLap rank fastestLapTime fastestLapSpeed  statusId  
0         39    2       1:27.452         218.300         1  
1         41    3       1:27.739         217.586       

In [4]:

master = pd.read_csv("output/phase1_master_data.csv")
master["date"] = pd.to_datetime(master["date"], errors="coerce")

# Keep modelling era
model_df = master[master["year"] >= 2000].copy()

# Clean positions
model_df["finish_position"] = pd.to_numeric(model_df["positionOrder"], errors="coerce")
model_df["quali_position"] = pd.to_numeric(model_df["quali_position"], errors="coerce")

# Sort chronologically
model_df = model_df.sort_values(["date", "raceId", "driverId"]).reset_index(drop=True)

# Driver and constructor season summaries
driver_year = (
    model_df.groupby(["driverId", "year"])
    .agg(
        driver_top3_pct=("top3_flag", "mean"),
        driver_avg_finish=("finish_position", "mean"),
        driver_quali_avg=("quali_position", "mean"),
        driver_quali_std=("quali_position", "std")
    )
    .reset_index()
)

constructor_year = (
    model_df.groupby(["constructorId", "year"])
    .agg(
        constructor_top3_pct=("top3_flag", "mean"),
        constructor_avg_finish=("finish_position", "mean")
    )
    .reset_index()
)

# Add lag features for years -1, -2, -3
for lag in [1, 2, 3]:
    tmp = driver_year.copy()
    tmp["year"] = tmp["year"] + lag
    tmp = tmp.rename(columns={
        "driver_top3_pct": f"driver_top3_pct_lag{lag}",
        "driver_avg_finish": f"driver_avg_finish_lag{lag}",
        "driver_quali_avg": f"driver_quali_avg_lag{lag}",
        "driver_quali_std": f"driver_quali_std_lag{lag}"
    })
    model_df = model_df.merge(tmp, on=["driverId", "year"], how="left")

for lag in [1, 2, 3]:
    tmp = constructor_year.copy()
    tmp["year"] = tmp["year"] + lag
    tmp = tmp.rename(columns={
        "constructor_top3_pct": f"constructor_top3_pct_lag{lag}",
        "constructor_avg_finish": f"constructor_avg_finish_lag{lag}"
    })
    model_df = model_df.merge(tmp, on=["constructorId", "year"], how="left")

# Qualifying consistency summary
model_df["quali_consistency_1y"] = model_df["driver_quali_std_lag1"]
model_df["quali_consistency_2y"] = model_df[
    ["driver_quali_std_lag1", "driver_quali_std_lag2"]
].mean(axis=1)
model_df["quali_consistency_3y"] = model_df[
    ["driver_quali_std_lag1", "driver_quali_std_lag2", "driver_quali_std_lag3"]
].mean(axis=1)

# Weather-specific last 3-year driver performance
weather_perf = (
    model_df.groupby(["driverId", "year", "weather_cat"])
    .agg(driver_weather_avg_finish=("finish_position", "mean"))
    .reset_index()
)

weather_features = []
for lag in [1, 2, 3]:
    tmp = weather_perf.copy()
    tmp["year"] = tmp["year"] + lag
    tmp = tmp.rename(columns={
        "driver_weather_avg_finish": f"driver_weather_avg_finish_lag{lag}"
    })
    weather_features.append(tmp)

weather_merged = weather_features[0]
for tmp in weather_features[1:]:
    weather_merged = weather_merged.merge(
        tmp, on=["driverId", "year", "weather_cat"], how="outer"
    )

weather_merged["driver_weather_avg_finish_last3y"] = weather_merged[
    [f"driver_weather_avg_finish_lag{i}" for i in [1, 2, 3]]
].mean(axis=1)

model_df = model_df.merge(
    weather_merged[["driverId", "year", "weather_cat", "driver_weather_avg_finish_last3y"]],
    on=["driverId", "year", "weather_cat"], how="left"
)

# Optional missing-value handling
feature_cols = [
    "driver_top3_pct_lag1", "driver_top3_pct_lag2", "driver_top3_pct_lag3",
    "constructor_top3_pct_lag1", "constructor_top3_pct_lag2", "constructor_top3_pct_lag3",
    "driver_avg_finish_lag1", "driver_avg_finish_lag2", "driver_avg_finish_lag3",
    "constructor_avg_finish_lag1", "constructor_avg_finish_lag2", "constructor_avg_finish_lag3",
    "quali_position", "quali_consistency_1y", "quali_consistency_2y", "quali_consistency_3y",
    "driver_weather_avg_finish_last3y"
]

for col in feature_cols:
    model_df[col] = model_df[col].fillna(model_df[col].median())

# Save final feature table
model_df.to_csv("phase2_model_data.csv", index=False)
print("Phase 2 complete.")

C:\Users\kpjm0\AppData\Local\Temp\ipykernel_38464\1613861318.py:1: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  master = pd.read_csv("output/phase1_master_data.csv")


Phase 2 complete.


In [5]:
from sklearn.metrics import (
    roc_auc_score, roc_curve, confusion_matrix, accuracy_score
)
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# -----------------------------
# Load model data
# -----------------------------
df = pd.read_csv("phase2_model_data.csv")
df["date"] = pd.to_datetime(df["date"], errors="coerce")

features = [
    "driver_top3_pct_lag1", "driver_top3_pct_lag2", "driver_top3_pct_lag3",
    "constructor_top3_pct_lag1", "constructor_top3_pct_lag2", "constructor_top3_pct_lag3",
    "driver_avg_finish_lag1", "driver_avg_finish_lag2", "driver_avg_finish_lag3",
    "constructor_avg_finish_lag1", "constructor_avg_finish_lag2", "constructor_avg_finish_lag3",
    "quali_position", "quali_consistency_1y", "quali_consistency_2y", "quali_consistency_3y",
    "driver_weather_avg_finish_last3y"
]

# Add weather dummies
weather_dummies = pd.get_dummies(df["weather_cat"], prefix="weather", drop_first=True)
df = pd.concat([df, weather_dummies], axis=1)
features = features + weather_dummies.columns.tolist()

# Drop rows with missing target
df = df.dropna(subset=["top3_flag"]).copy()

# -----------------------------
# Train-test split by season
# Example: train < 2023, test = 2023
# -----------------------------
train = df[df["year"] < 2023].copy()
test = df[df["year"] == 2023].copy()

X_train = train[features].copy()
y_train = train["top3_flag"].astype(int)

X_test = test[features].copy()
y_test = test["top3_flag"].astype(int)

# Scale numeric variables
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=features, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=features, index=X_test.index)

# -----------------------------
# VIF
# -----------------------------
X_vif = sm.add_constant(X_train_scaled)
vif_df = pd.DataFrame({
    "feature": X_vif.columns,
    "VIF": [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
})
print("\nVIF Table")
print(vif_df.sort_values("VIF", ascending=False))

# -----------------------------
# Logistic Regression via statsmodels
# -----------------------------
X_train_sm = sm.add_constant(X_train_scaled)
logit_model = sm.Logit(y_train, X_train_sm).fit()
print(logit_model.summary())

# Significant variable review
significance = pd.DataFrame({
    "feature": logit_model.params.index,
    "coef": logit_model.params.values,
    "p_value": logit_model.pvalues.values
})
print("\nSignificance table")
print(significance.sort_values("p_value"))

# -----------------------------
# Train predictions
# -----------------------------
train_probs = logit_model.predict(X_train_sm)
train_auc = roc_auc_score(y_train, train_probs)
print("\nTrain AUC:", train_auc)

fpr, tpr, thresholds = roc_curve(y_train, train_probs)
youden_j = tpr - fpr
best_idx = np.argmax(youden_j)
best_threshold = thresholds[best_idx]
print("Best threshold:", best_threshold)

train_pred = (train_probs >= best_threshold).astype(int)
train_cm = confusion_matrix(y_train, train_pred)
train_acc = accuracy_score(y_train, train_pred)

tn, fp, fn, tp = train_cm.ravel()
train_sens = tp / (tp + fn) if (tp + fn) else np.nan
train_spec = tn / (tn + fp) if (tn + fp) else np.nan

print("\nTrain confusion matrix")
print(train_cm)
print("Train accuracy:", train_acc)
print("Train sensitivity:", train_sens)
print("Train specificity:", train_spec)

# -----------------------------
# Test predictions
# -----------------------------
X_test_sm = sm.add_constant(X_test_scaled, has_constant="add")
test_probs = logit_model.predict(X_test_sm)
test_auc = roc_auc_score(y_test, test_probs)
print("\nTest AUC:", test_auc)

test_pred = (test_probs >= best_threshold).astype(int)
test_cm = confusion_matrix(y_test, test_pred)
test_acc = accuracy_score(y_test, test_pred)

tn, fp, fn, tp = test_cm.ravel()
test_sens = tp / (tp + fn) if (tp + fn) else np.nan
test_spec = tn / (tn + fp) if (tn + fp) else np.nan

print("\nTest confusion matrix")
print(test_cm)
print("Test accuracy:", test_acc)
print("Test sensitivity:", test_sens)
print("Test specificity:", test_spec)

# -----------------------------
# Predict podium for each 2023 race
# -----------------------------
test = test.copy()
test["pred_prob_top3"] = test_probs

predicted_podium_2023 = (
    test.sort_values(["raceId", "pred_prob_top3"], ascending=[True, False])
    .groupby("raceId")
    .head(3)
    .loc[:, ["raceId", "year", "round", "driverId", "driver_name", "constructor_name", "pred_prob_top3"]]
)

predicted_podium_2023.to_csv("phase3_predicted_top3_2023_logistic.csv", index=False)
vif_df.to_csv("phase3_vif_table.csv", index=False)
significance.to_csv("phase3_logit_significance.csv", index=False)

print("\nPhase 3 complete.")



VIF Table
                             feature       VIF
4          constructor_top3_pct_lag1  7.920235
5          constructor_top3_pct_lag2  7.592710
15              quali_consistency_2y  7.357647
1               driver_top3_pct_lag1  6.726433
6          constructor_top3_pct_lag3  6.618312
16              quali_consistency_3y  5.603870
10       constructor_avg_finish_lag1  5.590452
7             driver_avg_finish_lag1  5.566149
11       constructor_avg_finish_lag2  5.513495
2               driver_top3_pct_lag2  5.319223
12       constructor_avg_finish_lag3  4.997321
3               driver_top3_pct_lag3  4.355946
17  driver_weather_avg_finish_last3y  3.997381
8             driver_avg_finish_lag2  3.261156
9             driver_avg_finish_lag3  2.992384
14              quali_consistency_1y  2.488006
13                    quali_position  1.625938
20                       weather_wet  1.014887
18                   weather_unknown  1.014091
19                  weather_variable  1.012213
0 